# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library with all fields and entities referenced by their `@id` values.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded.")
print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, their `@id`s, and associated fields using the metadata.

In [ ]:
# List all record sets and, for each, show its @id and its fields' @id values
from mlcroissant.dataset.metadata import RecordSet

# The metadata.record_sets property provides the record set definitions
record_sets = metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"  Record Set Name: {rs.name}")
    print(f"    @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print(f"    Fields:")
        for field in rs.fields:
            print(f"      - {field.name} (@id: {field.id})")
    print()

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame.
We use the record set and field `@id` values as shown in the overview.

In [ ]:
# Extract dataframe for each record set using their @id
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))  # List of dicts keyed by field @id
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id} with shape {df.shape}")

# Display columns of the first record set, including their @id
if record_set_ids:
    primary_record_set_id = record_set_ids[0]
    print(f"\nColumns for record set '{primary_record_set_id}':")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())  # .head() to preview data

## 4. Exploratory Data Analysis (EDA)
Common data processing steps, such as filtering, normalizing, and grouping, can be performed using column `@id` values. For demonstration, we select a numeric field based on the previous list of columns.

In [ ]:
# --- Set up: choose a record set and its numeric field and group-by field ---
# Please replace with the actual @id values for your dataset as shown above
# For illustration, let's assume record set with index 0 is our main table

record_set_id = record_set_ids[0]  # Use first as the main table; adjust as desired
df = dataframes[record_set_id]

# Look for numeric fields by @id -- here, we select one by name or @id
print("All columns (by @id):", df.columns.tolist())

# If you know the numeric @id, set it below. Otherwise, here's how you might select one:
from pandas.api.types import is_numeric_dtype

numeric_candidates = [col for col in df.columns if is_numeric_dtype(df[col])]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # You may need to cast columns (uncomment and modify as needed)
    # Example: df['@id:age'] = pd.to_numeric(df['@id:age'], errors='coerce')
    numeric_field_id = df.columns[0]  # Fallback, may need update

print(f"Using numeric field @id: {numeric_field_id}")

# Demonstrate a simple filter
threshold = 10
if is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where `{numeric_field_id}` > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized `{numeric_field_id}` for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Column `{numeric_field_id}` is not numeric; please select/cast a numeric column.")

# Try grouping by another field
group_candidates = [col for col in df.columns if col != numeric_field_id]
group_field_id = None
for col in group_candidates:
    # Use a non-numeric or categorical column if available
    if not is_numeric_dtype(df[col]):
        group_field_id = col
        break

if group_field_id and is_numeric_dtype(filtered_df[numeric_field_id]):
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean of `{numeric_field_id}` by `{group_field_id}`:")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the primary record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field (if numeric)
if is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# If grouping field is available, plot boxplot
if group_field_id and is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} across {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, explore, and process the FAIR² dataset on clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors. 

- All entities, record sets, and fields were referenced by their schema `@id` values, ensuring reproducibility.
- We demonstrated loading the dataset, overviewing available record sets and fields, extracting data into DataFrames, basic analysis/filtering, and quick exploration visualizations.
- This approach can be adapted for further analytical or machine learning workflows on any Croissant-compatible dataset.

**Next steps:** Deepen your domain-specific analysis by examining relationships between fields of interest and leveraging external clinical/biological ontologies as needed.